In [1]:
#imports
import os, glob, json
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as sps

from enterprise.pulsar import Pulsar
import enterprise.signals.parameter as parameter
from enterprise.signals import utils
from enterprise.signals import signal_base
from enterprise.signals import selections
from enterprise.signals import white_signals
from enterprise.signals import gp_signals
from enterprise.signals import selections
from enterprise.signals.selections import Selection

from enterprise_extensions import models, hypermodel
from enterprise_extensions.model_utils import bayes_fac
from la_forge import core as co
from la_forge import diagnostics as dg

from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc

import pickle

Optional mpi4py package is not installed.  MPI support is not available.


# Get data

In [2]:
pkl_file = '/home/mitch/test_quick_burst/15_year_data/psrs_trimmed_SNR99p.pkl'
with open(pkl_file, 'rb') as file:
    psrs = pickle.load(file)

# Define output dir

In [3]:
parent_dir = '/home/mitch/test_quick_burst/15_year_data/15_year_CURN_run/'

# Define my model

In [12]:
psr = psrs[0]
selection = selections.Selection(selections.by_backend)
selection_ng = selections.Selection(selections.nanograv_backends)

#WN
efac = parameter.Uniform(0.01,10.0)
equad = parameter.Uniform(-8.5,-5)
ecorr = parameter.Uniform(-8.5,-5)

wn = white_signals.MeasurementNoise(efac=efac, log10_t2equad=equad)
ec = white_signals.EcorrKernelNoise(log10_ecorr=ecorr, selection=selection_ng)

#timing model
tm = gp_signals.MarginalizingTimingModel(use_svd=True)

#RN
log10_A = parameter.Uniform(-20,-11)("gw_curn_amp")
gamma = parameter.Uniform(0,7)('gw_gamma')

tmax = psr.toas.max()
tmin = psr.toas.min()
tspan = (tmax-tmin)
bins = 30
freq = np.arange(1/tspan, bins/tspan, 1/tspan)
pl = utils.powerlaw(log10_A=log10_A, gamma=gamma)
rn = gp_signals.FourierBasisGP(spectrum=pl, modes=freq)

#full model
s = wn + ec + tm + rn
pta = signal_base.PTA(s(psr))
x0 = np.hstack([p.sample() for p in pta.params])
ndim = len(x0)

cov = np.diag(np.ones(ndim)*0.01**2)

output_dir = parent_dir + psr.name
os.makedirs(output_dir, exist_ok=True)

sampler = ptmcmc(ndim, pta.get_lnlikelihood, pta.get_lnprior, cov, outDir=output_dir, resume=False)

sampler_steps = 10e5

sampler.sample(x0,sampler_steps, SCAMweight=30, AMweight=15, DEweight=50)

Package `fastshermanmorrison` not installed. Fallback to sherman-morrison


TypeError: unsupported operand type(s) for /: 'float' and 'MarginalizingNmat'